# HI-MoE Colab Runner

This notebook is a cleaned, runnable Colab workflow for the **HI-MoE** experiments.

## Before you run
1. In Colab, set **Runtime → Change runtime type → GPU**.
2. If available, set **Runtime version = 2025.07** (Python 3.11).
3. Put datasets in Google Drive under:

```text
MyDrive/HI_MOE/
├── datasets/
│   ├── coco/
│   ├── lvis/
│   └── objects365/
└── outputs/
    ├── work_dirs/
    ├── logs/
    └── figures/
```

This notebook:
- mounts Google Drive
- clones the GitHub repo
- patches the helper scripts
- installs a Colab-compatible OpenMMLab stack
- installs MMDetection from source
- copies HI-MoE modules into MMDetection
- generates configs
- patches dataset roots
- runs one pilot experiment

In [3]:
# ==== User settings ====
REPO_URL = "https://github.com/vashkelis/himoe.git"
REPO_BRANCH = "main"

DRIVE_ROOT = "/content/drive/MyDrive/HI_MOE"
DATASET_ROOT = f"{DRIVE_ROOT}/datasets"
OUTPUT_ROOT = f"{DRIVE_ROOT}/outputs"
WORK_REPO = "/content/hi_moe"
MMDET_ROOT = "/content/mmdetection"

# Default pilot config to run after setup
PILOT_CONFIG = "himoe_full_12e.py"

# Set to True if you also want the ablation grid cell at the end.
ENABLE_ABLATION_GRID = False

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pathlib

for p in [
    f"{DATASET_ROOT}/coco",
    f"{DATASET_ROOT}/lvis",
    f"{DATASET_ROOT}/objects365",
    f"{OUTPUT_ROOT}/work_dirs",
    f"{OUTPUT_ROOT}/logs",
    f"{OUTPUT_ROOT}/figures",
]:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)

Drive root: /content/drive/MyDrive/HI_MOE
Dataset root: /content/drive/MyDrive/HI_MOE/datasets
Output root: /content/drive/MyDrive/HI_MOE/outputs


In [5]:
import sys, torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "GPU is not enabled. In Colab, set Runtime → Change runtime type → GPU."
assert sys.version_info[:2] == (3, 11), "Use Colab runtime version 2025.07 (Python 3.11)."

python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
torch: 2.6.0+cu124
cuda available: True
cuda version: 12.4
gpu: Tesla T4


In [6]:
# Fresh clone of the HI-MoE repo
%cd /content
!rm -rf hi_moe
!git clone --branch "$REPO_BRANCH" "$REPO_URL" hi_moe
%cd /content/hi_moe
!git remote -v
!git branch --show-current
!find . -maxdepth 2 | sort

/content
Cloning into 'hi_moe'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 51 (delta 20), reused 47 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 15.71 KiB | 15.71 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/hi_moe
origin	https://github.com/vashkelis/himoe.git (fetch)
origin	https://github.com/vashkelis/himoe.git (push)
main
.
./configs
./configs/generated
./configs/himoe_config_template.py
./CONTRIBUTING.md
./.git
./.git/branches
./.git/config
./.git/description
./.git/HEAD
./.git/hooks
./.gitignore
./.git/index
./.git/info
./.git/logs
./.git/objects
./.git/packed-refs
./.git/refs
./HI_MoE_Colab.ipynb
./projects
./projects/hi_moe
./README.md
./requirements.txt
./scripts
./scripts/generate_configs.py
./scripts/patch_dataset_root.py
./scripts/run_ablation_grid.py
./scripts/setup_colab.sh


## Patch the helper scripts

The current repository scripts contain some environment-specific paths and an older install flow.  
This cell rewrites them into Colab-safe versions.

In [7]:
%%writefile /content/hi_moe/scripts/generate_configs.py
from pathlib import Path

ROOT = Path('/content/hi_moe')
template_path = ROOT / 'configs' / 'himoe_config_template.py'
out_dir = ROOT / 'configs' / 'generated'
template = template_path.read_text()
out_dir.mkdir(parents=True, exist_ok=True)

variants = {
    'himoe_full_12e.py': dict(num_experts=16, num_scene_groups=4, topk=2, scene_topk=2,
                              use_scene_routing='True', use_instance_routing='True', use_shared_expert='True',
                              apply_encoder_moe='True', apply_decoder_moe='True', max_epochs=12,
                              lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_token_moe_12e.py': dict(num_experts=16, num_scene_groups=1, topk=2, scene_topk=1,
                                   use_scene_routing='False', use_instance_routing='True', use_shared_expert='True',
                                   apply_encoder_moe='True', apply_decoder_moe='True', max_epochs=12,
                                   lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_instance_only_12e.py': dict(num_experts=16, num_scene_groups=1, topk=2, scene_topk=1,
                                       use_scene_routing='False', use_instance_routing='True', use_shared_expert='True',
                                       apply_encoder_moe='False', apply_decoder_moe='True', max_epochs=12,
                                       lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_scene_only_12e.py': dict(num_experts=16, num_scene_groups=4, topk=1, scene_topk=2,
                                    use_scene_routing='True', use_instance_routing='False', use_shared_expert='True',
                                    apply_encoder_moe='False', apply_decoder_moe='True', max_epochs=12,
                                    lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_k1_12e.py': dict(num_experts=16, num_scene_groups=4, topk=1, scene_topk=2,
                            use_scene_routing='True', use_instance_routing='True', use_shared_expert='True',
                            apply_encoder_moe='True', apply_decoder_moe='True', max_epochs=12,
                            lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_k4_12e.py': dict(num_experts=16, num_scene_groups=4, topk=4, scene_topk=2,
                            use_scene_routing='True', use_instance_routing='True', use_shared_expert='True',
                            apply_encoder_moe='True', apply_decoder_moe='True', max_epochs=12,
                            lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_e4_12e.py': dict(num_experts=4, num_scene_groups=2, topk=2, scene_topk=1,
                            use_scene_routing='True', use_instance_routing='True', use_shared_expert='True',
                            apply_encoder_moe='True', apply_decoder_moe='True', max_epochs=12,
                            lambda_balance=0.01, lambda_diversity=0.001),
    'himoe_e8_12e.py': dict(num_experts=8, num_scene_groups=4, topk=2, scene_topk=2,
                            use_scene_routing='True', use_instance_routing='True', use_shared_expert='True',
                            apply_encoder_moe='True', apply_decoder_moe='True', max_epochs=12,
                            lambda_balance=0.01, lambda_diversity=0.001),
}

for name, vals in variants.items():
    text = template
    for k, v in vals.items():
        text = text.replace('{{ ' + k + ' }}', str(v))
    (out_dir / name).write_text(text)
    print('wrote', out_dir / name)

Overwriting /content/hi_moe/scripts/generate_configs.py


In [8]:
%%writefile /content/hi_moe/scripts/patch_dataset_root.py
import argparse
from pathlib import Path

def patch_config(cfg_path: Path, dataset: str, data_root: str):
    text = cfg_path.read_text()
    if 'data_root =' in text:
        print(f'skip {cfg_path} (already patched)')
        return
    if dataset == 'coco':
        patch = f"""

data_root = '{data_root.rstrip('/')}/'
train_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        ann_file='annotations/instances_train2017.json',
        data_prefix=dict(img='train2017/')))
val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        ann_file='annotations/instances_val2017.json',
        data_prefix=dict(img='val2017/')))
test_dataloader = val_dataloader
val_evaluator = dict(ann_file=data_root + 'annotations/instances_val2017.json')
test_evaluator = val_evaluator
"""
    else:
        patch = f"""

data_root = '{data_root.rstrip('/')}/'
"""
    cfg_path.write_text(text + patch)
    print('patched', cfg_path)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--dataset', default='coco')
    parser.add_argument('--data-root', required=True)
    args = parser.parse_args()
    patch_config(Path(args.config), args.dataset, args.data_root)

if __name__ == '__main__':
    main()

Overwriting /content/hi_moe/scripts/patch_dataset_root.py


In [9]:
%%writefile /content/hi_moe/scripts/run_ablation_grid.py
import argparse
import subprocess
from pathlib import Path

VARIANTS = [
    'himoe_full_12e.py',
    'himoe_token_moe_12e.py',
    'himoe_instance_only_12e.py',
    'himoe_scene_only_12e.py',
    'himoe_k1_12e.py',
    'himoe_k4_12e.py',
    'himoe_e4_12e.py',
    'himoe_e8_12e.py',
]

def run(cmd, cwd=None):
    print('RUN', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--mmdet-root', default='/content/mmdetection')
    parser.add_argument('--variant-dir', default='/content/mmdetection/projects/hi_moe')
    parser.add_argument('--work-base', default='/content/drive/MyDrive/HI_MOE/outputs/work_dirs')
    parser.add_argument('--amp', action='store_true')
    args = parser.parse_args()

    Path(args.work_base).mkdir(parents=True, exist_ok=True)

    for variant in VARIANTS:
        cfg = f'{args.variant_dir}/{variant}'
        work_dir = f'{args.work_base}/{Path(variant).stem}'
        cmd = ['python', 'tools/train.py', cfg, '--work-dir', work_dir]
        if args.amp:
            cmd.append('--amp')
        run(cmd, cwd=args.mmdet_root)

if __name__ == '__main__':
    main()

Overwriting /content/hi_moe/scripts/run_ablation_grid.py


## Install the OpenMMLab stack

This flow uses a Colab-friendly combination:
- PyTorch 2.1.0
- MMCV 2.1.0 wheel for cu121 / torch2.1.0
- MMDetection from source, installed without build isolation

# **Use only one cell below, not two:**

**A: Fresh runtime install**
Use this after:
- Runtime → Factory reset runtime
- changing runtime to GPU
- starting from a clean Colab session

In [ ]:
# =========================
# Fresh runtime install
# =========================

# 1) Colab-friendly packaging/tooling
!python -m pip install -U "pip<26" "setuptools<82" wheel "packaging>=23.2,<25"
!python -m pip install -U requests==2.32.3 jedi==0.19.2 "tqdm>=4.66.3" "rich>=13.7.1,<14"

# 2) Remove only the OpenMMLab / torch stack we are about to control
!python -m pip uninstall -y torch torchvision torchaudio mmcv mmcv-lite mmengine mmdet openmim mim || true

# 3) Install a known-good torch stack for OpenMMLab
!python -m pip install --no-cache-dir \
  torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0

# 4) OpenMMLab core
!python -m pip install -U mmengine

# 5) MMCV from matching wheel index
!python -m pip install \
  mmcv==2.1.0 \
  -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html

# 6) Sanity check
import sys, torch, requests, jedi
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("requests:", requests.__version__)
print("jedi:", jedi.__version__)

**B: Resume runtime install**

Use this when:
- you already have a working Colab session
- you do not want to wipe torch/MMCV/MMDetection
- you just want to ensure the required packages are present

In [ ]:
# =========================
# Resume runtime install
# =========================

# 1) Keep Colab-friendly utility versions
!python -m pip install -U "pip<26" "setuptools<82" wheel "packaging>=23.2,<25"
!python -m pip install -U requests==2.32.3 jedi==0.19.2 "tqdm>=4.66.3" "rich>=13.7.1,<14"

# 2) Install only missing OpenMMLab pieces if needed
!python - <<'PY'
import importlib.util

mods = {
    "torch": "torch",
    "mmengine": "mmengine",
    "mmcv": "mmcv",
    "mmdet": "mmdet",
}
for name, mod in mods.items():
    print(name, "OK" if importlib.util.find_spec(mod) is not None else "MISSING")
PY

# 3) Install only what is missing
import importlib.util, os

need_mmengine = importlib.util.find_spec("mmengine") is None
need_mmcv = importlib.util.find_spec("mmcv") is None

if need_mmengine:
    os.system("python -m pip install -U mmengine")

if need_mmcv:
    os.system(
        "python -m pip install "
        "mmcv==2.1.0 "
        "-f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html"
    )

# 4) Sanity check
import sys, requests, jedi
print("python:", sys.version)
print("requests:", requests.__version__)
print("jedi:", jedi.__version__)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda version:", torch.version.cuda)
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch check failed:", e)

for mod_name in ["mmengine", "mmcv", "mmdet"]:
    try:
        mod = __import__(mod_name)
        print(f"{mod_name}:", getattr(mod, "__version__", "unknown"))
    except Exception as e:
        print(f"{mod_name} not importable:", e)

In [10]:
# Base packaging tools: keep them compatible with Colab
!python -m pip install -U "pip<26" "setuptools<82" wheel "packaging>=23.2,<25"

# Colab-friendly utility versions
!python -m pip install -U requests==2.32.3 jedi==0.19.2 "tqdm>=4.66.3" "rich>=13.7.1,<14"

# OpenMMLab core
!python -m pip install -U mmengine

# MMCV: use the wheel index matching torch 2.1.0 / cu121 only if you already downgraded torch
!python -m pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html

Found existing installation: transformers 4.53.2
Uninstalling transformers-4.53.2:
  Successfully uninstalled transformers-4.53.2
Found existing installation: accelerate 1.8.1
Uninstalling accelerate-1.8.1:
  Successfully uninstalled accelerate-1.8.1
Found existing installation: peft 0.16.0
Uninstalling peft-0.16.0:
  Successfully uninstalled peft-0.16.0
Found existing installation: sentence-transformers 4.1.0
Uninstalling sentence-transformers-4.1.0:
  Successfully uninstalled sentence-transformers-4.1.0
Found existing installation: tokenizers 0.21.2
Uninstalling tokenizers-0.21.2:
  Successfully uninstalled tokenizers-0.21.2
Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Succe

In [ ]:
import sys, torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Install MMDetection from source
%cd /content
!rm -rf mmdetection
!git clone https://github.com/open-mmlab/mmdetection.git
%cd /content/mmdetection
!python -m pip install -r requirements/build.txt
!python -m pip install -v -e . --no-build-isolation

In [ ]:
# Verify MMDetection stack
import mmcv, mmengine, mmdet
print("mmcv:", mmcv.__version__)
print("mmengine:", mmengine.__version__)
print("mmdet:", mmdet.__version__)

## Prepare HI-MoE modules inside MMDetection

The template config imports `projects.hi_moe.*`, so the easiest Colab setup is
to copy the repo's HI-MoE package and generated configs into `mmdetection/projects/hi_moe`.

In [ ]:
# Copy HI-MoE project files into MMDetection's projects folder
!mkdir -p /content/mmdetection/projects/hi_moe
!cp -r /content/hi_moe/projects/hi_moe/* /content/mmdetection/projects/hi_moe/
!touch /content/mmdetection/projects/__init__.py
!touch /content/mmdetection/projects/hi_moe/__init__.py

# Generate configs in the repo
%cd /content/hi_moe
!python scripts/generate_configs.py

# Patch each generated config for Drive-backed COCO
for cfg in pathlib.Path('/content/hi_moe/configs/generated').glob('*.py'):
    !python /content/hi_moe/scripts/patch_dataset_root.py --config "$cfg" --dataset coco --data-root "$DATASET_ROOT/coco"

# Copy generated configs into MMDetection
!cp /content/hi_moe/configs/generated/*.py /content/mmdetection/projects/hi_moe/
!find /content/mmdetection/projects/hi_moe -maxdepth 1 -type f | sort

In [ ]:
# Final import check
import sys
for p in ['/content/mmdetection', '/content/hi_moe']:
    if p not in sys.path:
        sys.path.insert(0, p)

from mmdet.registry import MODELS
from projects.hi_moe.himoe_ffn import HiMoEFFN
print("HiMoEFFN import OK")

## Run one pilot experiment

This uses the generated config selected in `PILOT_CONFIG`.
Logs and checkpoints are saved to Google Drive.

In [ ]:
%cd /content/mmdetection
pilot_cfg = f"/content/mmdetection/projects/hi_moe/{PILOT_CONFIG}"
pilot_workdir = f"{OUTPUT_ROOT}/work_dirs/{pathlib.Path(PILOT_CONFIG).stem}"

print("Config:", pilot_cfg)
print("Work dir:", pilot_workdir)
!python tools/train.py "$pilot_cfg" --work-dir "$pilot_workdir"

## Optional: evaluate the latest checkpoint

Uncomment and run this after training if you want COCO bbox evaluation.

In [ ]:
# %cd /content/mmdetection
# ckpt = f"{OUTPUT_ROOT}/work_dirs/{pathlib.Path(PILOT_CONFIG).stem}/latest.pth"
# cfg = f"/content/mmdetection/projects/hi_moe/{PILOT_CONFIG}"
# !python tools/test.py "$cfg" "$ckpt" --eval bbox

## Optional: run the ablation grid

Set `ENABLE_ABLATION_GRID = True` in the first cell if you want this.

In [ ]:
if ENABLE_ABLATION_GRID:
    %cd /content/hi_moe
    !python scripts/run_ablation_grid.py
else:
    print("ENABLE_ABLATION_GRID is False. Skipping.")